# Visualization Notebook (Sphere Flow)
Includes 2D slices and an interactive 3D Q-criterion view (drag to rotate).


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import yt
from skimage import measure


In [ ]:
# --- Settings ---
plot_root = Path('out_sphere_full')
plotfiles = sorted(plot_root.glob('plt*'))
if not plotfiles:
    raise FileNotFoundError('No plotfiles found in out_sphere_full')
preferred = plot_root / 'plt03600'
plotfile = preferred if preferred.is_dir() else plotfiles[-1]
stride = 2  # downsample for 3D speed; increase for detail
output_dir = Path('_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)
thresholds_file = output_dir / 'q_thresholds.json'
isos_fixed = [2.5e-08, 5.0e-08, 5.0e-07]
colors = ['#cfcfcf', '#4a90e2', '#e74c3c']  # grey, blue, red
alphas = [0.35, 0.25, 0.25]
sphere_color = '#4a4a4a'
sphere_alpha = 1.0
view_azim = 119.4
view_elev = 9.0
view_dist = 1.51


In [ ]:
def load_velocity(plotfile: Path):
    ds = yt.load(str(plotfile))
    base_dims = np.array(ds.domain_dimensions, dtype=int)
    max_lev = int(ds.index.max_level)
    dims = base_dims * (int(ds.refine_by) ** max_lev)
    cg = ds.covering_grid(level=max_lev, left_edge=ds.domain_left_edge, dims=dims, num_ghost_zones=0)
    ux = np.array(cg[("boxlib", "ux")], dtype=np.float64)
    uy = np.array(cg[("boxlib", "uy")], dtype=np.float64)
    uz = np.array(cg[("boxlib", "uz")], dtype=np.float64)
    lo = np.array(ds.domain_left_edge, dtype=np.float64)
    hi = np.array(ds.domain_right_edge, dtype=np.float64)
    return ux, uy, uz, float(ds.current_time), lo, hi

def compute_qcriterion(ux, uy, uz, dx, dy, dz):
    du_dx, du_dy, du_dz = np.gradient(ux, dx, dy, dz, edge_order=2)
    dv_dx, dv_dy, dv_dz = np.gradient(uy, dx, dy, dz, edge_order=2)
    dw_dx, dw_dy, dw_dz = np.gradient(uz, dx, dy, dz, edge_order=2)
    s11 = du_dx; s22 = dv_dy; s33 = dw_dz
    s12 = 0.5 * (du_dy + dv_dx)
    s13 = 0.5 * (du_dz + dw_dx)
    s23 = 0.5 * (dv_dz + dw_dy)
    o12 = 0.5 * (du_dy - dv_dx)
    o13 = 0.5 * (du_dz - dw_dx)
    o23 = 0.5 * (dv_dz - dw_dy)
    s2 = s11*s11 + s22*s22 + s33*s33 + 2.0*(s12*s12 + s13*s13 + s23*s23)
    o2 = 2.0 * (o12*o12 + o13*o13 + o23*o23)
    return 0.5 * (o2 - s2)

def load_sphere_params(inputs_path=Path('inputs')):
    vals = {}
    if not inputs_path.exists():
        return None
    for line in inputs_path.read_text().splitlines():
        line = line.strip()
        if line.startswith('ibm.x0'): vals['x0'] = float(line.split('=')[1])
        if line.startswith('ibm.y0'): vals['y0'] = float(line.split('=')[1])
        if line.startswith('ibm.z0'): vals['z0'] = float(line.split('=')[1])
        if line.startswith('ibm.R'):  vals['R'] = float(line.split('=')[1])
    if all(k in vals for k in ('x0','y0','z0','R')):
        return vals['x0'], vals['y0'], vals['z0'], vals['R']
    return None

def load_u0(inputs_path=Path('inputs')):
    if not inputs_path.exists():
        return None
    for line in inputs_path.read_text().splitlines():
        line = line.strip()
        if line.startswith('lbmPhysicalParameters.U0'):
            return float(line.split('=')[1])
    return None


## 2D Mid-plane Slices


In [ ]:
ux, uy, uz, t, lo, hi = load_velocity(plotfile)
mid_k = ux.shape[2] // 2
speed = np.sqrt(ux[:, :, mid_k] ** 2 + uy[:, :, mid_k] ** 2 + uz[:, :, mid_k] ** 2)
# computed vorticity from ux, uy
dx = (hi[0] - lo[0]) / float(ux.shape[0])
dy = (hi[1] - lo[1]) / float(ux.shape[1])
dz = (hi[2] - lo[2]) / float(ux.shape[2])
dux_dx, dux_dy, _ = np.gradient(ux, dx, dy, dz, edge_order=2)
duy_dx, _, _ = np.gradient(uy, dx, dy, dz, edge_order=2)
vor = duy_dx[:, :, mid_k] - dux_dy[:, :, mid_k]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
extent = [float(lo[0]), float(hi[0]), float(lo[1]), float(hi[1])]
im0 = axes[0].imshow(vor.T, origin='lower', extent=extent, cmap='RdBu_r')
axes[0].set_title(f'Vorticity (mid-z), t={t:.1f}')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
fig.colorbar(im0, ax=axes[0], shrink=0.85, label='vor')
im1 = axes[1].imshow(speed.T, origin='lower', extent=extent, cmap='viridis')
axes[1].set_title(f'Speed |u| (mid-z), t={t:.1f}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
fig.colorbar(im1, ax=axes[1], shrink=0.85, label='|u|')
plt.show()

fig.savefig(output_dir / 'sphere_slices.png', dpi=150)
print(f"Saved {output_dir / 'sphere_slices.png'}")


## Interactive 3D Q-criterion


In [ ]:
# Build meshes
if stride > 1:
    ux3 = ux[::stride, ::stride, ::stride]
    uy3 = uy[::stride, ::stride, ::stride]
    uz3 = uz[::stride, ::stride, ::stride]
else:
    ux3, uy3, uz3 = ux, uy, uz

nx, ny, nz = ux3.shape
dx = (hi[0] - lo[0]) / float(nx)
dy = (hi[1] - lo[1]) / float(ny)
dz = (hi[2] - lo[2]) / float(nz)
q = compute_qcriterion(ux3, uy3, uz3, dx, dy, dz)

# Thresholds (keep consistent for movies)
isos = isos_fixed
thresholds_file.write_text(json.dumps({'method': 'q', 'isos': isos, 'iso_min_fraction': 0.05}))

scale = np.array([(hi[0] - lo[0]) / (q.shape[0] - 1),
                  (hi[1] - lo[1]) / (q.shape[1] - 1),
                  (hi[2] - lo[2]) / (q.shape[2] - 1)])

meshes = []
for iso, color, alpha in sorted(zip(isos, colors, alphas), key=lambda t: t[0]):
    verts, faces, _, _ = measure.marching_cubes(q, level=iso)
    verts = verts * scale + lo
    meshes.append(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color, opacity=alpha, flatshading=True, name=f'Q={iso:.2e}'
    ))

sphere = load_sphere_params()
u0 = load_u0()
if sphere is not None:
    x0, y0, z0, R = sphere
    u = np.linspace(0.0, 2.0*np.pi, 64)
    v = np.linspace(0.0, np.pi, 32)
    xs = x0 + R * np.outer(np.cos(u), np.sin(v))
    ys = y0 + R * np.outer(np.sin(u), np.sin(v))
    zs = z0 + R * np.outer(np.ones_like(u), np.cos(v))
    # Solid sphere mesh for reliable visibility
    verts = np.column_stack([xs.ravel(), ys.ravel(), zs.ravel()])
    nu, nv = xs.shape
    faces_i = []; faces_j = []; faces_k = []
    for i in range(nu - 1):
        for j in range(nv - 1):
            idx = i * nv + j
            idx_i2 = (i + 1) * nv + j
            idx_j2 = i * nv + (j + 1)
            idx_i2_j2 = (i + 1) * nv + (j + 1)
            faces_i.extend([idx, idx])
            faces_j.extend([idx_i2, idx_i2_j2])
            faces_k.extend([idx_i2_j2, idx_j2])
    sphere_mesh = go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2], i=faces_i, j=faces_j, k=faces_k, color=sphere_color, opacity=sphere_alpha, name='sphere', hoverinfo='skip')
else:
    sphere_mesh = None

# Triad bottom-right and scale bar upstream away from sphere
lines = []
labels = []
xr, yr, zr = (hi - lo)
triad = np.array([hi[0] - 0.08*xr, lo[1] + 0.08*yr, lo[2] + 0.08*zr])
triad_len = 0.12 * min(xr, yr, zr)
ox, oy, oz = triad
lines.extend([
    go.Scatter3d(x=[ox, ox+triad_len], y=[oy, oy], z=[oz, oz], mode='lines', line=dict(color='#444444', width=5), showlegend=False),
    go.Scatter3d(x=[ox, ox], y=[oy, oy+triad_len], z=[oz, oz], mode='lines', line=dict(color='#444444', width=5), showlegend=False),
    go.Scatter3d(x=[ox, ox], y=[oy, oy], z=[oz, oz+triad_len], mode='lines', line=dict(color='#444444', width=5), showlegend=False),
])
labels.extend([
    go.Scatter3d(x=[ox + triad_len*1.35], y=[oy], z=[oz], mode='text', text=['x'], textfont=dict(color='#444444', size=12), showlegend=False),
    go.Scatter3d(x=[ox], y=[oy + triad_len*1.35], z=[oz], mode='text', text=['y'], textfont=dict(color='#444444', size=12), showlegend=False),
    go.Scatter3d(x=[ox], y=[oy], z=[oz + triad_len*1.35], mode='text', text=['z'], textfont=dict(color='#444444', size=12), showlegend=False),
])

# Scale bar removed per request.

t_label = f't={t:.1f}'
if sphere is not None and u0 is not None and R > 0:
    t_conv = t * u0 / (2.0 * R)
    t_label = f'tU/D = {t_conv:.2f}'

fig = go.Figure(data=[*meshes, *lines, *labels] + ([sphere_mesh] if sphere_mesh is not None else []))
import math
az = math.radians(view_azim)
el = math.radians(view_elev)
eye = [math.cos(el) * math.cos(az) * view_dist, math.cos(el) * math.sin(az) * view_dist, math.sin(el) * view_dist]
fig.update_layout(
    title=f'Q-criterion isosurfaces ({t_label})',
    scene=dict(
        aspectmode='data',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        camera=dict(eye=dict(x=eye[0], y=eye[1], z=eye[2])),
    ),
    width=900,
    height=520,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
)
fig.write_html(output_dir / 'sphere_q_plt03600.html', include_plotlyjs=True)
print(f"Saved {output_dir / 'sphere_q_plt03600.html'}")
fig.show()


# Generated Figures (tU/D = 7.2)

**Mid-plane slices:**

![Mid-plane slices](_artifacts/sphere_slices.png)

**Q-criterion (PNG):**

![Q-criterion](_artifacts/sphere_q_plt03600.png)

**Interactive 3D HTML:**

[_artifacts/sphere_q_plt03600.html](_artifacts/sphere_q_plt03600.html)

**Movie:**

`_artifacts/sphere_q_all.mp4`
